
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Lecture - MLflow's Evaluation Framework

## Overview

MLflow provides a comprehensive evaluation framework specifically designed for generative AI applications, offering automated judges, tracing capabilities, and systematic assessment tools. This lecture explores MLflow's architecture, core components, and how they work together to enable rigorous agent evaluation.

We'll examine the three fundamental components of MLflow evaluation, understand the `mlflow.genai.evaluate()` function, and explore how tracing provides the foundation for sophisticated evaluation capabilities.

**Learning Objectives**

By the end of this lecture, you will be able to:
- Describe MLflow's evaluation framework architecture and key components
- Understand the role of evaluation datasets, scorers, and predict functions
- Explain how the `mlflow.genai.evaluate()` function orchestrates evaluation
- Understand the role of tracing in agent evaluation and debugging
- Recognize how AI Gateway integration enables production monitoring

## A. MLflow Overview and OpenTelemetry Integration

### A1. MLflow Platform Overview

![preparing-for-evaluation.png](../Includes/images/Evaluation with MLflow/preparing-for-evaluation.png "preparing-for-evaluation.png")

MLflow is an open source platform for managing the full machine learning lifecycle, and it now natively supports OpenTelemetry for advanced observability and monitoring, allowing seamless tracing and metrics export to external observability platforms.

**MLflow Core Features:**
- Experiment tracking with parameters, metrics, and model lineage
- Model Registry for versioning, stage transitions, and annotations (integrated with Unity Catalog)
- Model deployment for batch, streaming, and real-time inference
- Real-time tracing server (MLflow Tracing) for instant observability
- Production monitoring including automatic scoring of GenAI traces
- Deep support for prompt engineering and GenAI evaluation workflows

### A2. OpenTelemetry Integration

**OpenTelemetry Integration Benefits:**
- OpenTelemetry is an open standard for telemetry data collection and export, widely adopted for observability across cloud-native systems
- MLflow traces are fully compatible with OpenTelemetry trace specifications, allowing export to popular solutions (e.g., Datadog, New Relic, Grafana, Splunk)
- MLflow supports three trace export modes:
  - MLflow tracking only (default): sends traces to MLflow Tracking Server
  - OpenTelemetry only: sends traces to an OpenTelemetry Collector
  - Dual export: sends traces to both MLflow and an OpenTelemetry Collector

## B. Core Architecture

### B1. Three Fundamental Components

<div style="text-align: center; line-height: 0; padding-top: 9px;">
<img src="https://docs.databricks.com/aws/en/assets/images/flowchart-00c729ac75207b58d9c2243583a30d5a.png" alt="MLFlow Evaluation">
</div>

MLflow provides a comprehensive evaluation framework designed specifically for generative AI applications. The architecture centers on three fundamental components:

### B2. Component 1: Evaluation Datasets

Your evaluation dataset defines what you're testing. At minimum, it contains inputs (queries or requests to your agent). Optionally, it can include:

- **Outputs**: Pre-generated agent responses for faster evaluation without re-running inference
- **Expectations**: Ground truth information such as expected facts, expected responses, or per-row guidelines
- **Traces**: Complete execution traces for analyzing multi-step agent behavior
- **Metadata**: Additional context like user preferences, conversation history, or retrieved documents

Datasets are typically stored as JSON files or Pandas DataFrames for easy manipulation and versioning.

### B3. Component 2: Scorers (Judges)

Scorers evaluate your agent's outputs against defined criteria. MLflow provides multiple scorer types:

- **Built-in judges**: Research-validated LLM-based assessments for common criteria like correctness, relevance, and safety
- **Guideline judges**: Custom business rules expressed in natural language
- **Code-based scorers**: Python functions for deterministic evaluation (length checks, format validation, etc.)
- **Custom LLM judges**: Your own LLM-based evaluation logic for specialized requirements

### B4. Component 3: Predict Function

The predict function generates outputs for your evaluation dataset. This can be:

- Your agent's prediction method (for evaluating on-the-fly)
- A lambda that transforms inputs to the format your agent expects
- Omitted entirely if you're evaluating pre-generated outputs

These three components come together in `mlflow.genai.evaluate()`, which orchestrates the evaluation process, collects metrics, and logs comprehensive results for analysis.

## C. The `mlflow.genai.evaluate()` Function

### C1. Central Orchestration Point

```python
import mlflow
from mlflow.genai.scorers import Correctness

results = mlflow.genai.evaluate(
    data=eval_dataset,                  # DataFrame, list[dict], or EvaluationDataset
    scorers=[Correctness()],            # Built-in and/or custom scorers
    predict_fn=my_app,                  # Optional: direct evaluation
    # model_id="models:/my-app/1",      # Optional: link to versioned app
)
```

The `mlflow.genai.evaluate()` function serves as the central orchestration point for agent evaluation. Understanding its behavior is critical for effective evaluation workflows.

### C2. Key Parameters

**Key parameters:**
- `data`: Your evaluation dataset
- `scorers`: List of scoring functions
- `predict_fn`: Your agent/app function
- `model_id`: Optional model reference

### C3. Evaluation Workflow

**Evaluation workflow:**

1. **Data loading**: MLflow loads your evaluation dataset and validates its structure
2. **Output generation**: If `predict_fn` is provided, MLflow calls it for each input to generate outputs
3. **Trace creation**: Each prediction creates an MLflow trace when your predict_fn is instrumented (e.g., with `@mlflow.trace` or `mlflow.openai.autolog`) or when evaluating endpoints with `mlflow.genai.to_predict_fn`. For "answer sheet" mode, evaluate constructs traces from inputs/outputs even without running the app
4. **Scorer execution**: Each scorer evaluates the inputs/outputs/traces according to its logic
5. **Result aggregation**: Individual scores are aggregated into summary metrics
6. **Logging**: Results are logged to MLflow for analysis and comparison

### C4. Return Value and Results Access

**Return value:**

The function returns an `EvaluationResult` object containing:
- **run_id**: Unique identifier for this evaluation run
- **metrics**: Aggregated metrics across all examples (e.g., average score, pass rate)

**Accessing per-example results:**

In MLflow 3, per-example results are accessed via traces rather than a result_df attribute:
```python
eval_traces = mlflow.search_traces(run_id=results.run_id)
```

This structured approach enables systematic evaluation while maintaining full observability into individual agent interactions.

## D. MLflow Tracing: The Foundation of Agent Observability

### D1. Comprehensive Observability

MLflow Tracing provides comprehensive observability into agent execution, capturing every step of your agent's reasoning process. Tracing is enabled when using `mlflow.genai.evaluate()` with properly instrumented predict functions and forms the foundation for many evaluation capabilities.

**What tracing captures:**

- **Model calls**: Every interaction with foundation models, including prompts, responses, and model parameters
- **Tool invocations**: Function calls with input parameters and return values
- **Retrieval operations**: Documents retrieved from vector stores, with content and metadata
- **Timing information**: Duration of each operation for performance analysis
- **Hierarchical structure**: Parent-child relationships showing the execution flow

### D2. Span Types in Traces

MLflow organizes traces into spans with specific types:
- **Root span**: Top-level span representing the complete agent invocation
- **RETRIEVER**: Spans where documents are fetched from vector search or other retrieval systems
- **TOOL**: Individual tool or function calls
- **CHAT_MODEL**: Language model interactions
- **CHAIN**: Sequences of operations (common in LangChain-based agents)

### D3. Why Tracing Matters for Evaluation

**Why tracing matters for evaluation:**

Certain evaluation judges, like `RetrievalSufficiency`, require traces to function. They analyze what was retrieved (not just the final response) to assess whether the retrieval system provided adequate information. Without traces, these sophisticated evaluations would be impossible.

Tracing also enables debugging by allowing you to inspect exactly what happened during a failed evaluation, identifying whether issues stem from retrieval quality, tool selection, or LLM reasoning.

## E. AI Gateway Integration and Production Monitoring

### E1. AI Gateway Integration

When you register agents using the Mosaic AI Agent Framework and deploy them to Model Serving, Databricks automatically enables AI Gateway-enhanced inference tables. These tables provide detailed logging of all requests and responses in production.

**Inference table benefits:**

- **Automatic logging**: Every request to your deployed agent is captured without additional instrumentation
- **Rich metadata**: Includes request/response content, timestamps, latency, model versions, and trace data
- **Query interface**: SQL-queryable tables in Unity Catalog for analysis and monitoring
- **Evaluation integration**: Inference table data can be directly used as evaluation datasets

### E2. From Production to Evaluation

This integration creates a powerful feedback loop:
1. Agents run in production and log all interactions to inference tables
2. You query inference tables to extract interesting examples, failure cases, or edge cases
3. These real-world examples augment your evaluation dataset
4. Future evaluations test against actual production scenarios

This approach ensures your evaluation dataset evolves with real user behavior rather than remaining static and potentially stale.

## F. Key Takeaways

MLflow's evaluation framework provides a comprehensive solution for agent evaluation through:

1. **Three-component architecture**: Evaluation datasets, scorers, and predict functions work together seamlessly
2. **Central orchestration**: `mlflow.genai.evaluate()` handles the complexity of evaluation workflows
3. **Comprehensive tracing**: Full observability into agent execution enables sophisticated evaluation and debugging
4. **Production integration**: AI Gateway and inference tables create feedback loops between production and evaluation
5. **OpenTelemetry compatibility**: Integration with industry-standard observability tools

This foundation enables systematic, scalable evaluation that grows with your agent development needs. The next lectures will explore the specific types of judges and evaluation strategies you can implement within this framework.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>